# J5: Seed ensemble + conformal
Train a >=4-seed ensemble, average quantiles, then re-fit conformal on the ENSEMBLE output (never inherited from a member).

In [ ]:
import sys; sys.path.insert(0, '../../src')
import os; os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import pandas as pd, numpy as np, torch
from data.loaders import load_split
from module_joint.config import JointConfig, select_device
LQ = pd.read_parquet('../../data/module_a/load_quantiles.parquet')
device = select_device(); print('device', device)


In [ ]:
from module_joint.ensemble import train_ensemble
from module_joint.calibrate import conformalize
from module_joint.evaluate import compare_price, make_truth
train, val = load_split('train'), load_split('val')
val_ctx = pd.concat([train.tail(168), val])
cfg = JointConfig(backbone='tide', max_epochs=80)
ens = train_ensemble(cfg, seeds=[0,1,2,3], train_df=train, val_df=val, load_quantiles=LQ, device=device)
pr = ens.predict_quantiles(val_ctx, LQ, restrict_to=val.index)['price']
tr = make_truth(val_ctx, pr.index, 'price')
print('raw', compare_price(pr, tr)['overall'])
